In [0]:
# ============================================================================
# SETUP: Install Required Dependencies
# ============================================================================
# This notebook predicts in-hospital mortality from MIMIC-III discharge summaries
# using two approaches: TF-IDF + Logistic Regression (baseline) and DistilBERT (deep learning)

!pip install -q scikit-learn matplotlib seaborn tqdm      # ML, visualization, progress bars
!pip install -q transformers datasets accelerate          # HuggingFace ecosystem for BERT
!pip install -q sentencepiece                             # Tokenizer support for transformer models


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Restart the Python kernel to ensure newly installed packages are loaded properly
dbutils.library.restartPython()

In [0]:
# ============================================================================
# IMPORTS: Load All Required Libraries
# ============================================================================

# Standard libraries
import os
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm  # Progress bars for long operations

# Scikit-learn: Traditional ML pipeline
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, roc_auc_score,
                             roc_curve, confusion_matrix, accuracy_score)
from sklearn.utils import class_weight  # Handle imbalanced classes

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# HuggingFace: Deep learning NLP
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import Dataset, DatasetDict  # Efficient dataset handling for transformers


2026-03-15 04:36:25.445154: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-15 04:36:25.464670: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-15 04:36:25.470614: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-15 04:36:25.485683: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-15 04:36:29.547331: W tensorflow/compiler/tf2

In [0]:
# ============================================================================
# DATA LOADING: Read MIMIC-III Clinical Data
# ============================================================================
# - NOTEEVENTS: Contains clinical notes (discharge summaries, radiology reports, etc.)
# - ADMISSIONS: Contains admission info including mortality outcome (HOSPITAL_EXPIRE_FLAG)

notes = pd.read_csv('./MIMIC-III/NOTEEVENTS.csv', low_memory=False)
admissions = pd.read_csv('./MIMIC-III/ADMISSIONS.csv', low_memory=False)

print("NOTEEVENTS shape:", notes.shape)
print("ADMISSIONS shape:", admissions.shape)


NOTEEVENTS shape: (134745, 11)
ADMISSIONS shape: (58976, 19)


In [0]:
# ============================================================================
# PREPROCESSING: Extract and Clean Discharge Summaries
# ============================================================================
# Discharge summaries contain comprehensive patient information at the end of stay,
# making them ideal for mortality prediction tasks

# Filter to only discharge summaries (most complete clinical documentation)
ds = notes[notes['CATEGORY'] == 'Discharge summary'].copy()

# Keep only essential columns and remove rows with missing admission ID or text
ds = ds[['SUBJECT_ID', 'HADM_ID', 'CHARTDATE', 'CATEGORY', 'TEXT']].dropna(subset=['HADM_ID','TEXT'])
ds['HADM_ID'] = ds['HADM_ID'].astype(int)

def clean_text(text):
    """
    Clean clinical text by removing de-identification markers and normalizing whitespace.
    MIMIC-III uses [**...**] brackets to mark redacted PHI (dates, names, etc.)
    """
    text = re.sub(r'\[\*\*.*?\*\*\]', ' ', str(text))  # Remove de-identification markers
    text = re.sub(r'\s+', ' ', text)                    # Normalize whitespace
    return text.strip()

ds['TEXT_CLEAN'] = ds['TEXT'].map(clean_text)
print("Discharge summaries:", ds.shape[0])
ds.head(2)


Discharge summaries: 49011


,SUBJECT_ID,HADM_ID,CHARTDATE,CATEGORY,TEXT,TEXT_CLEAN
0,22532,167853,8/4/2151,Discharge summary,Admission Date: [**2151-7-16**] Dischar...,Admission Date: Discharge Date: Service: ADDEN...
1,13702,107527,6/14/2118,Discharge summary,Admission Date: [**2118-6-2**] Discharg...,Admission Date: Discharge Date: Date of Birth:...


In [0]:
# ============================================================================
# LABEL CREATION: Merge Notes with Mortality Outcomes
# ============================================================================
# HOSPITAL_EXPIRE_FLAG indicates if the patient died during the hospital stay
# 1 = died in hospital (positive class), 0 = survived (negative class)

# Ensure matching data types for merge
admissions['HADM_ID'] = admissions['HADM_ID'].astype(int)

# Join discharge summaries with admission outcomes
merged = ds.merge(
    admissions[['HADM_ID', 'HOSPITAL_EXPIRE_FLAG', 'ADMISSION_TYPE', 'ADMITTIME', 'DISCHTIME']],
    on='HADM_ID', 
    how='left'
)

# Drop records without outcome labels and create binary target variable
merged = merged.dropna(subset=['HOSPITAL_EXPIRE_FLAG'])
merged['label'] = merged['HOSPITAL_EXPIRE_FLAG'].astype(int)

print("Total labeled notes:", merged.shape[0])
print("Positive (in-hospital death):", merged['label'].sum())


Total labeled notes: 49011
Positive (in-hospital death): 4736


In [0]:
# ============================================================================
# DATA SPLITTING: Create Train/Validation/Test Sets
# ============================================================================
# Stratified split preserves the class ratio (~9.7% positive) across all sets
# Split ratios: ~72% train, ~13% validation, ~15% test

train_df, test_df = train_test_split(
    merged, test_size=0.15, random_state=42, stratify=merged['label']
)
train_df, val_df = train_test_split(
    train_df, test_size=0.15, random_state=42, stratify=train_df['label']
)

print(f"Train: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")


35410 6249 7352


In [0]:
# Verify dataset sizes (using full dataset without subsampling)
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


Train: 35410
Validation: 6249
Test: 7352


In [0]:
# ============================================================================
# CLASS BALANCE CHECK: Verify Stratification Worked
# ============================================================================
# All splits should have similar class distribution (~90% negative, ~10% positive)
# This imbalanced ratio reflects real-world mortality rates

print("Train class distribution:")
print(train_df["label"].value_counts(normalize=True))
print("\nValidation class distribution:")
print(val_df["label"].value_counts(normalize=True))
print("\nTest class distribution:")
print(test_df["label"].value_counts(normalize=True))


0    0.903361
1    0.096639
Name: label, dtype: float64
0    0.903345
1    0.096655
Name: label, dtype: float64
0    0.903428
1    0.096572
Name: label, dtype: float64


In [0]:
# ============================================================================
# BASELINE MODEL: TF-IDF + Logistic Regression
# ============================================================================
# A fast, interpretable baseline using bag-of-words features
# TF-IDF captures word importance relative to the document and corpus

# Create TF-IDF feature matrix
# - max_features: Limit vocabulary to top 20K terms to reduce noise
# - ngram_range: Include both single words and word pairs (bigrams)
# - stop_words: Remove common English words that don't carry meaning
vectorizer = TfidfVectorizer(
    max_features=20000, 
    ngram_range=(1,2), 
    stop_words='english'
)

# Fit vectorizer on training data only to prevent data leakage
X_train = vectorizer.fit_transform(train_df['TEXT_CLEAN'])
X_val = vectorizer.transform(val_df['TEXT_CLEAN'])
X_test = vectorizer.transform(test_df['TEXT_CLEAN'])

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

# Address class imbalance by computing balanced weights
# Gives higher weight to minority class (deaths) during training
weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = {i: w for i, w in enumerate(weights)}

# Train logistic regression classifier
# - saga solver: Efficient for large datasets, supports L1/L2 regularization
# - n_jobs=-1: Use all CPU cores for parallel processing
clf = LogisticRegression(max_iter=1000, class_weight=class_weights, solver='saga', n_jobs=-1)
clf.fit(X_train, y_train)

# Generate predictions and probability scores for ROC-AUC
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:,1]  # Probability of positive class (death)

print(classification_report(y_test, y_pred, digits=4))
print("ROC AUC:", roc_auc_score(y_test, y_proba))


Uploading artifacts:   0%|          | 0/3 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0     0.9959    0.9824    0.9891      6642
           1     0.8538    0.9620    0.9046       710

    accuracy                         0.9804      7352
   macro avg     0.9248    0.9722    0.9469      7352
weighted avg     0.9822    0.9804    0.9809      7352

ROC AUC: 0.9960250815340704


In [0]:
# ============================================================================
# MODEL INTERPRETABILITY: Examine Top Predictive Features
# ============================================================================
# Logistic regression coefficients reveal which words/phrases predict mortality
# Positive coefficients = associated with death, Negative = associated with survival

feature_names = np.array(vectorizer.get_feature_names_out())
coefs = clf.coef_[0]

# Sort features by coefficient magnitude
top_pos_idx = np.argsort(coefs)[-50:][::-1]  # Strongest positive predictors
top_neg_idx = np.argsort(coefs)[:50]          # Strongest negative predictors

print("Top positive predictors (associated with in-hospital death):")
for idx in top_pos_idx[:20]:
    print(f"  {feature_names[idx]}: {coefs[idx]:.4f}")


Top positive predictors:
expired 19.4883
expired discharge 10.6934
deceased 7.2253
disposition expired 7.0469
death 6.9591
family 6.6544
passed away 4.6971
deceased discharge 4.6027
autopsy 4.4421
measures 4.3084
patient expired 4.2674
instructions followup 4.1376
condition expired 4.046
dead 4.0057
away 3.9326
despite 3.7945
comfort 3.7097
comfort measures 3.6625
condition deceased 3.5947
cmo 3.5914


In [0]:
# ============================================================================
# DEEP LEARNING: BERT-based Classification
# ============================================================================
# Transformer models capture contextual word relationships that TF-IDF misses
# Trade-off: Better understanding of language nuances vs. higher compute cost



In [0]:
# ============================================================================
# DATA PREPARATION: Convert to HuggingFace Dataset Format
# ============================================================================
# HuggingFace Datasets provide efficient batching, caching, and memory mapping
# Required column names: 'text' for input, 'label' for target

train_hf = Dataset.from_pandas(train_df[['TEXT_CLEAN','label']].rename(columns={'TEXT_CLEAN':'text'}))
val_hf = Dataset.from_pandas(val_df[['TEXT_CLEAN','label']].rename(columns={'TEXT_CLEAN':'text'}))
test_hf = Dataset.from_pandas(test_df[['TEXT_CLEAN','label']].rename(columns={'TEXT_CLEAN':'text'}))

# Combine into a DatasetDict for easy access during training
dataset = DatasetDict({'train': train_hf, 'validation': val_hf, 'test': test_hf})


In [0]:
# ============================================================================
# TOKENIZATION: Convert Text to Model Input Format
# ============================================================================
# DistilBERT: A distilled (compressed) version of BERT
# - ~2x faster training with ~97% of BERT's performance
# - Same tokenizer and architecture, fewer transformer layers

MODEL_NAME = "distilbert-base-uncased"  
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_fn(batch):
    """
    Tokenize text for transformer input.
    - truncation: Cut text exceeding max_length (important for long clinical notes)
    - padding: Pad shorter sequences to uniform length for batching
    - max_length: Set to 128 tokens for faster training (trade-off with coverage)
    """
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

# Apply tokenization to all dataset splits (batched for efficiency)
tokenized = dataset.map(preprocess_fn, batched=True)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/35410 [00:00<?, ? examples/s]

Map:   0%|          | 0/6249 [00:00<?, ? examples/s]

Map:   0%|          | 0/7352 [00:00<?, ? examples/s]

In [0]:
# Verify tokenized dataset sizes match original dataframes
print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))


Train samples: 35410
Validation samples: 6249
Test samples: 7352


In [0]:
# ============================================================================
# MODEL TRAINING: Fine-tune DistilBERT for Mortality Prediction
# ============================================================================
# Using HuggingFace Trainer API (Transformers v5) for streamlined training workflow
# Run this after tokenized DatasetDict is created and validated

import os
import torch
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

# 1) Clear distributed training environment variables
# Prevents accidental multi-GPU distributed training on single GPU setup
for k in ["RANK","WORLD_SIZE","LOCAL_RANK","MASTER_ADDR","MASTER_PORT","NODE_RANK"]:
    os.environ.pop(k, None)
os.environ["WORLD_SIZE"] = "1"
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "-1"

# 2) Prepare labels column - HuggingFace Trainer expects 'labels' (not 'label')
if "tokenized" not in globals():
    raise RuntimeError("tokenized not found — create tokenized DatasetDict before running this cell.")

for split in tokenized.keys():
    ds = tokenized[split]
    if "labels" not in ds.column_names:
        if "label" in ds.column_names:
            tokenized[split] = ds.rename_column("label", "labels")
        else:
            raise RuntimeError(f"Dataset split '{split}' has no 'label' or 'labels' column. Columns: {ds.column_names}")

print("Columns in train split:", tokenized["train"].column_names)

# 3) Define evaluation metrics (called automatically by Trainer at each epoch)
def compute_metrics(eval_pred):
    logits, labels = eval_pred # eval_pred contains the model outputs (logits) and the true labels
    
    # Convert raw model scores (logits) into predicted class labels
    preds = np.argmax(logits, axis=-1)
    
    # Compute overall accuracy: proportion of correct predictions
    acc = accuracy_score(labels, preds)
    
    # Compute precision, recall, and F1 score for the binary classification task
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    
    try:
        # Convert logits to probabilities using the softmax function
        probs = torch.softmax(torch.tensor(logits), dim=1)[:,1].numpy()
        
        # Compute ROC-AUC using the probability of the positive class (death)
        roc_auc = roc_auc_score(labels, probs)
        
    except Exception:
        # return NaN instead of crashing
        roc_auc = float("nan")
    
    return {     # Return all computed evaluation metrics as a dictionary
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc
    }

# 4) Load pre-trained model and configure training hyperparameters
MODEL_NAME = globals().get("MODEL_NAME", "distilbert-base-uncased")
print("Loading model:", MODEL_NAME)
# Load DistilBERT with a classification head (2 output neurons for binary classification)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).cuda()

training_args = TrainingArguments(
    output_dir="/tmp/bert_mimic",        # Directory for checkpoints (if saving)
    eval_strategy="epoch",               # Evaluate after each epoch
    save_strategy="no",                  # Don't save checkpoints (saves disk space)
    per_device_train_batch_size=32,      # Samples per GPU per forward pass
    per_device_eval_batch_size=32,
    num_train_epochs=3,                  # Total training passes through dataset
    weight_decay=0.01,                   # L2 regularization to prevent overfitting
    logging_steps=50,                    # Log metrics every 50 steps
    dataloader_num_workers=0,            # Data loading workers (0 = main process)
    fp16=True,                           # Mixed precision for faster training
    optim="adamw_torch",                 # Adam optimizer with weight decay
    report_to=[],                        # Disable external logging (W&B, etc.)
    push_to_hub=False,
)

# 5) Create Trainer and link model, data, and metrics together
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics
)

# 6) Start training and evaluate
print("Device: cuda")
print("Starting training...")
train_result = trainer.train()
print("Training finished. Train metrics:", train_result.metrics)

# Final evaluation on validation set
eval_metrics = trainer.evaluate(eval_dataset=tokenized["validation"])
print("Validation metrics:", eval_metrics)


Columns in train split: ['text', 'labels', '__index_level_0__', 'input_ids', 'attention_mask']
Loading model: distilbert-base-uncased


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[2026-03-15 04:40:03,987] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


Device: cuda
Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.271600,0.267577,0.907985,0.723077,0.077815,0.140508,0.805810
2,0.252400,0.263917,0.909105,0.743243,0.091060,0.162242,0.832742
3,0.200600,0.257514,0.909906,0.582996,0.238411,0.338425,0.835186


Training finished. Train metrics: {'train_runtime': 333.0379, 'train_samples_per_second': 318.973, 'train_steps_per_second': 9.972, 'total_flos': 3518002939806720.0, 'train_loss': 0.24881657373829802, 'epoch': 3.0}


Validation metrics: {'eval_loss': 0.25751376152038574, 'eval_accuracy': 0.909905584893583, 'eval_precision': 0.582995951417004, 'eval_recall': 0.23841059602649006, 'eval_f1': 0.3384253819036428, 'eval_roc_auc': 0.8351861519600654, 'eval_runtime': 5.3564, 'eval_samples_per_second': 1166.635, 'eval_steps_per_second': 36.592, 'epoch': 3.0}


In [0]:
# ============================================================================
# TEST SET EVALUATION: Final Performance Assessment
# ============================================================================
# Evaluate on held-out test set to get unbiased performance estimate
# This should only be done once to avoid overfitting to the test set

metrics = trainer.evaluate(tokenized['test'])
print("Test set metrics:", metrics)


{'eval_loss': 0.27518025040626526, 'eval_accuracy': 0.9005712731229597, 'eval_precision': 0.4668769716088328, 'eval_recall': 0.2084507042253521, 'eval_f1': 0.2882181110029211, 'eval_roc_auc': 0.8128590573855661, 'eval_runtime': 6.8361, 'eval_samples_per_second': 1075.465, 'eval_steps_per_second': 33.645, 'epoch': 3.0}


In [0]:
# ============================================================================
# MODEL PERSISTENCE: Save Baseline Model for Deployment
# ============================================================================
# Save the TF-IDF vectorizer and logistic regression model for later use
# Both are needed for inference: vectorizer transforms text, model predicts

import joblib
joblib.dump(vectorizer, 'tfidf_vectorizer.joblib')  # Saves vocabulary and IDF weights
joblib.dump(clf, 'logreg_model.joblib')             # Saves trained model coefficients

['logreg_model.joblib']

In [0]:
# ============================================================================
# DATA LEAKAGE ANALYSIS: Check for Outcome Terms in Text
# ============================================================================
# ISSUE: Discharge summaries are written AFTER the outcome is known,
# so they often explicitly mention "expired" or "died" for deceased patients.
# This causes artificially inflated model performance (the model learns to 
# detect outcome phrases rather than clinical risk factors).
#
# For a truly predictive model, you would need to Use notes written BEFORE the outcome 

# Common terms that directly reveal the patient outcome
outcome_terms = ["expired", "died", "death", "pronounced dead", "dead on arrival", 
                 "passed away", "expired on", "dead"]

def contains_outcome(text):
    """Check if text contains any outcome-revealing terms."""
    t = str(text).lower()
    return any(term in t for term in outcome_terms)

# Compare prevalence of outcome terms between positive and negative classes
pos = merged[merged['label']==1]['TEXT_CLEAN']  # Patients who died
neg = merged[merged['label']==0]['TEXT_CLEAN']  # Patients who survived

pos_frac = pos.map(contains_outcome).mean()
neg_frac = neg.map(contains_outcome).mean()

# High positive fraction + low negative fraction = strong evidence of data leakage
print(f"Fraction of positive notes containing outcome terms: {pos_frac:.3f}")
print(f"Fraction of negative notes containing outcome terms: {neg_frac:.3f}")


Fraction of positive notes containing outcome terms: 0.932
Fraction of negative notes containing outcome terms: 0.200
